# Barter-RS: Instruments & Market Data

This notebook covers the full Rust data layer: instruments, indexed lookups,
trade streaming, multi-exchange composition, L1/L2 order books, and low-level
WebSocket integration traits.

## Topics Covered
- Core types: `ExchangeId`, `Asset`, `Instrument`, `InstrumentKind`
- `IndexedInstruments` for O(1) indexed lookups
- The `Keyed<Key, Value>` utility pattern
- Streaming public trades from Binance Futures
- L1 order book streaming (best bid/ask)
- L2 order book manager with `OrderBookMap`
- Multi-exchange, multi-kind unified streams with `DataKind`
- `Transformer`, `Validator`, `ExchangeStream` traits
- Error classification: `Unrecoverable` and `Terminal`


In [ ]:
:dep barter-data = { version = "0.11" }
:dep barter-instrument = { version = "0.3" }
:dep barter-integration = { version = "0.10", features = ["channel", "collection", "error", "protocol", "serde", "subscription", "stream"] }
:dep tokio = { version = "1", features = ["full"] }
:dep tokio-stream = "0.1"
:dep tokio-tungstenite = { version = "0.26", features = ["url", "rustls-tls-webpki-roots"] }
:dep futures = "0.3"
:dep futures-util = "0.3"
:dep serde = { version = "1.0", features = ["derive"] }
:dep serde_json = "1.0"
:dep tracing = "0.1"
:dep tracing-subscriber = { version = "0.3", features = ["env-filter"] }


---
## Part 1: Instruments & Index Management


## 1. Core Types

Barter uses strongly-typed wrappers for all financial primitives:

- **`ExchangeId`** — identifies an exchange (e.g., `BinanceSpot`, `Okx`)
- **`Asset`** — a tradeable asset (e.g., `btc`, `usdt`)
- **`Instrument`** — a specific trading pair on an exchange
- **`Underlying`** — the base/quote pair (e.g., BTC/USDT)
- **`InstrumentKind`** — Spot, Perpetual, Future, or Option

In [ ]:
use barter_instrument::{
    Underlying,
    exchange::ExchangeId,
    instrument::{Instrument, InstrumentIndex, kind::InstrumentKind},
    asset::AssetIndex,
    index::IndexedInstruments,
};

// Define instruments across multiple exchanges and kinds
let instruments = IndexedInstruments::builder()
    // Binance Spot BTC/USDT
    .add_instrument(Instrument::spot(
        ExchangeId::BinanceSpot,
        "binance_spot_btc_usdt",
        "BTCUSDT",
        Underlying::new("btc", "usdt"),
        None,
    ))
    // Binance Spot ETH/USDT
    .add_instrument(Instrument::spot(
        ExchangeId::BinanceSpot,
        "binance_spot_eth_usdt",
        "ETHUSDT",
        Underlying::new("eth", "usdt"),
        None,
    ))
    // OKX Spot BTC/USDT
    .add_instrument(Instrument::spot(
        ExchangeId::Okx,
        "okx_spot_btc_usdt",
        "BTC-USDT",
        Underlying::new("btc", "usdt"),
        None,
    ))
    .build();

println!("Built IndexedInstruments:");
println!("  Exchanges: {}", instruments.exchanges().len());
println!("  Assets:    {}", instruments.assets().len());
println!("  Instruments: {}", instruments.instruments().len());

## 2. Indexed Lookups

All entities are assigned compact integer indices at build time. This enables
O(1) array-based lookups instead of HashMap access — critical for hot-path
trading logic.

```
ExchangeId::BinanceSpot  →  ExchangeIndex(0)
ExchangeId::Okx          →  ExchangeIndex(1)
"btc"                    →  AssetIndex(0)
"usdt"                   →  AssetIndex(1)
"BTCUSDT" on Binance     →  InstrumentIndex(0)
```

In [ ]:
// Access instruments by index
println!("Instruments by index:");
for (idx, instrument) in instruments.instruments().iter().enumerate() {
    println!(
        "  InstrumentIndex({idx}): exchange={}, name={}, kind={:?}, base={}, quote={}",
        instrument.exchange,
        instrument.name_exchange,
        instrument.kind,
        instrument.underlying.base,
        instrument.underlying.quote,
    );
}

println!("\nAssets by index:");
for (idx, asset) in instruments.assets().iter().enumerate() {
    println!("  AssetIndex({idx}): {asset:?}");
}

println!("\nExchanges by index:");
for (idx, exchange) in instruments.exchanges().iter().enumerate() {
    println!("  ExchangeIndex({idx}): {exchange:?}");
}

## 3. Instrument Kinds

Barter supports multiple instrument kinds for different market types:

In [ ]:
use barter_instrument::instrument::kind::InstrumentKind;

// Available instrument kinds
let kinds = [
    InstrumentKind::Spot,
    InstrumentKind::Perpetual,
];

println!("Supported InstrumentKinds:");
for kind in &kinds {
    println!("  - {kind:?}");
}

println!("\nAdditionally: Future (with expiry) and Option (with strike, expiry, option_kind)");

## 4. JSON Serialisation

Instruments serialise cleanly to JSON, which is how `SystemConfig` stores them.

In [ ]:
let instrument = Instrument::spot(
    ExchangeId::BinanceSpot,
    "binance_spot_btc_usdt",
    "BTCUSDT",
    Underlying::new("btc", "usdt"),
    None,
);

let json = serde_json::to_string_pretty(&instrument)
    .expect("failed to serialise");

println!("Instrument as JSON:\n{json}");

// Round-trip deserialisation
let roundtrip: Instrument = serde_json::from_str(&json)
    .expect("failed to deserialise");

println!("\nRound-trip successful: {} on {}", roundtrip.name_exchange, roundtrip.exchange);

## 5. The `Keyed<Key, Value>` Pattern

Barter uses `Keyed<K, V>` throughout to associate values with their index keys.
This is a simple but powerful pattern for maintaining type-safe key-value
associations without using maps.

```rust
pub struct Keyed<Key, Value> {
    pub key: Key,
    pub value: Value,
}
```

This is used for:
- `Keyed<InstrumentIndex, Position>` — a position tagged with its instrument
- `Keyed<AssetIndex, Balance>` — a balance tagged with its asset
- `Keyed<ExchangeIndex, Vec<Order>>` — orders grouped by exchange

In [ ]:
use barter_instrument::Keyed;
use barter_instrument::instrument::InstrumentIndex;

// Example: tagging a value with its instrument index
let position_pnl = Keyed {
    key: InstrumentIndex(0),
    value: 1500.0_f64, // PnL in quote currency
};

println!(
    "Instrument {:?} has PnL: {:.2}",
    position_pnl.key, position_pnl.value
);

## Architecture: Why Indexed?

Traditional trading systems use `HashMap<String, T>` for lookups. Barter replaces
this with indexed arrays:

| Approach | Lookup Cost | Cache Behaviour | Memory |
|----------|-------------|-----------------|--------|
| `HashMap<String, T>` | O(1) amortised, hash + compare | Cache-unfriendly (pointer chasing) | Higher (string allocs) |
| `Vec<T>` + `Index(usize)` | O(1) guaranteed | Cache-friendly (contiguous) | Minimal |

The `IndexedInstruments` builder resolves all string identifiers to compact
indices at startup. All hot-path code then uses `ExchangeIndex`, `AssetIndex`,
and `InstrumentIndex` for direct array access.

---
## Part 2: Market Data Streaming


## Setup

First, add the required dependencies. Since evcxr resolves crates from crates.io, we reference the published versions.

## 1. Streaming Public Trades

The `Streams` builder provides a fluent API for subscribing to market data.
Each `.subscribe()` call creates a **separate WebSocket connection**, which is useful
for isolating high-volume instruments.

In [ ]:
use barter_data::{
    exchange::binance::futures::BinanceFuturesUsd,
    streams::{Streams, reconnect::stream::ReconnectingStream},
    subscription::trade::PublicTrades,
};
use barter_instrument::instrument::market_data::kind::MarketDataInstrumentKind;
use futures_util::StreamExt;

// Build streams for BTC and ETH perpetual futures on Binance
let streams = Streams::<PublicTrades>::builder()
    // High-volume: dedicated WebSocket for BTC
    .subscribe([
        (BinanceFuturesUsd::default(), "btc", "usdt", MarketDataInstrumentKind::Perpetual, PublicTrades),
    ])
    // High-volume: dedicated WebSocket for ETH
    .subscribe([
        (BinanceFuturesUsd::default(), "eth", "usdt", MarketDataInstrumentKind::Perpetual, PublicTrades),
    ])
    // Lower volume instruments can share a single connection
    .subscribe([
        (BinanceFuturesUsd::default(), "sol", "usdt", MarketDataInstrumentKind::Perpetual, PublicTrades),
        (BinanceFuturesUsd::default(), "avax", "usdt", MarketDataInstrumentKind::Perpetual, PublicTrades),
    ])
    .init()
    .await
    .expect("failed to initialise streams");

println!("Streams initialised! Reading first 5 trades...");

// Merge all exchange streams and consume events
let mut joined = streams
    .select_all()
    .with_error_handler(|error| eprintln!("Stream error: {error:?}"));

let mut count = 0;
while let Some(event) = joined.next().await {
    println!(
        "[{}] {} {} | price={} qty={}",
        event.exchange, event.instrument, event.kind.side,
        event.kind.price, event.kind.quantity
    );
    count += 1;
    if count >= 5 { break; }
}

## 2. Streaming L1 Order Books (Best Bid/Ask)

L1 order book data provides the best bid and ask prices — useful for spread monitoring
and simple execution logic.

In [ ]:
use barter_data::{
    exchange::binance::spot::BinanceSpot,
    streams::{Streams, reconnect::stream::ReconnectingStream},
    subscription::book::OrderBooksL1,
};
use barter_instrument::instrument::market_data::kind::MarketDataInstrumentKind;
use futures_util::StreamExt;

let streams = Streams::<OrderBooksL1>::builder()
    .subscribe([
        (BinanceSpot::default(), "btc", "usdt", MarketDataInstrumentKind::Spot, OrderBooksL1),
        (BinanceSpot::default(), "eth", "usdt", MarketDataInstrumentKind::Spot, OrderBooksL1),
    ])
    .init()
    .await
    .expect("failed to initialise L1 streams");

println!("L1 OrderBook streams initialised! Reading first 5 updates...");

let mut joined = streams
    .select_all()
    .with_error_handler(|error| eprintln!("Stream error: {error:?}"));

let mut count = 0;
while let Some(event) = joined.next().await {
    println!(
        "[{}] {} | best_bid={} best_ask={}",
        event.exchange, event.instrument,
        event.kind.best_bid.price, event.kind.best_ask.price
    );
    count += 1;
    if count >= 5 { break; }
}

## 3. L2 Order Book Manager

For deeper market analysis, `barter-data` provides a managed L2 order book that
automatically applies incremental updates. The `OrderBookMap` gives thread-safe
read access to the latest snapshot at any time.

In [ ]:
use barter_data::{
    books::{manager::init_multi_order_book_l2_manager, map::OrderBookMap},
    exchange::binance::spot::BinanceSpot,
    subscription::book::OrderBooksL2,
};
use barter_instrument::instrument::market_data::{
    MarketDataInstrument, kind::MarketDataInstrumentKind,
};
use std::time::Duration;

// Initialise L2 OrderBook manager with separate connections per instrument group
let book_manager = init_multi_order_book_l2_manager([
    vec![
        (BinanceSpot::default(), "btc", "usdt", MarketDataInstrumentKind::Spot, OrderBooksL2),
    ],
    vec![
        (BinanceSpot::default(), "eth", "usdt", MarketDataInstrumentKind::Spot, OrderBooksL2),
    ],
]).await.expect("failed to init L2 book manager");

// Clone the book map for concurrent read access
let books = book_manager.books.clone();

// Spawn the manager to apply incremental updates in the background
tokio::spawn(book_manager.run());

// Wait for some updates to arrive, then read snapshots
let instrument = MarketDataInstrument::new("btc", "usdt", MarketDataInstrumentKind::Spot);

tokio::time::sleep(Duration::from_secs(3)).await;
if let Some(book) = books.find(&instrument) {
    let snapshot = book.read();
    println!("BTC/USDT L2 OrderBook snapshot:");
    println!("  Bids (top 3): {:?}", snapshot.bids.iter().take(3).collect::<Vec<_>>());
    println!("  Asks (top 3): {:?}", snapshot.asks.iter().take(3).collect::<Vec<_>>());
} else {
    println!("Book not yet available");
}

## Key Concepts

| Concept | Description |
|---------|-------------|
| `Streams::builder()` | Fluent API entry point. Each `.subscribe()` opens a new WebSocket. |
| `ReconnectingStream` | Wraps streams with automatic reconnection on transient errors. |
| `select_all()` | Merges all exchange streams into a single unified stream. |
| `with_error_handler()` | Attaches an error callback without terminating the stream. |
| `OrderBookMap` | Thread-safe (`Arc<RwLock>`) local order book with snapshot reads. |

### Subscription Types

- `PublicTrades` — individual trade executions
- `OrderBooksL1` — best bid/ask (top of book)
- `OrderBooksL2` — full depth order book with incremental updates

---
## Part 3: Multi-Exchange Data Streams


## 1. Building Multi-Kind, Multi-Exchange Streams

The `Streams::builder_multi()` API composes multiple subscription-type builders
into a single `Streams<MarketStreamResult<_, DataKind>>`. This normalises all
event types into the `DataKind` enum, making it easy to handle heterogeneous data
in one event loop.

In [ ]:
use barter_data::{
    event::DataKind,
    exchange::{
        binance::{futures::BinanceFuturesUsd, spot::BinanceSpot},
        okx::Okx,
    },
    streams::{Streams, consumer::MarketStreamResult, reconnect::stream::ReconnectingStream},
    subscription::{
        book::{OrderBooksL1, OrderBooksL2},
        trade::PublicTrades,
    },
};
use barter_instrument::instrument::market_data::{
    MarketDataInstrument, kind::MarketDataInstrumentKind,
};
use tokio_stream::StreamExt;

// Build a combined stream across exchanges and data kinds
let streams: Streams<MarketStreamResult<MarketDataInstrument, DataKind>> = Streams::builder_multi()
    // PublicTrades from multiple exchanges
    .add(Streams::<PublicTrades>::builder()
        .subscribe([
            (BinanceSpot::default(), "btc", "usdt", MarketDataInstrumentKind::Spot, PublicTrades),
        ])
        .subscribe([
            (BinanceFuturesUsd::default(), "btc", "usdt", MarketDataInstrumentKind::Perpetual, PublicTrades),
        ])
        .subscribe([
            (Okx, "btc", "usdt", MarketDataInstrumentKind::Spot, PublicTrades),
        ])
    )
    // OrderBooksL1 from Binance
    .add(Streams::<OrderBooksL1>::builder()
        .subscribe([
            (BinanceSpot::default(), "btc", "usdt", MarketDataInstrumentKind::Spot, OrderBooksL1),
        ])
    )
    .init()
    .await
    .expect("failed to init multi-exchange streams");

println!("Multi-exchange streams initialised!");
println!("Reading first 10 events across all exchanges and data types...\n");

## 2. Consuming the Unified Stream

All events are normalised into `MarketEvent<_, DataKind>`. Pattern-match on
`DataKind` to handle each subscription type.

In [ ]:
let mut joined = streams
    .select_all()
    .with_error_handler(|error| eprintln!("Stream error: {error:?}"));

let mut count = 0;
while let Some(event) = joined.next().await {
    match &event.kind {
        DataKind::Trade(trade) => {
            println!(
                "[TRADE]  {:>15} {:>20} | side={:>4} price={:<12} qty={}",
                event.exchange, event.instrument,
                format!("{}", trade.side), trade.price, trade.quantity
            );
        }
        DataKind::OrderBookL1(book) => {
            println!(
                "[L1]     {:>15} {:>20} | bid={:<12} ask={}",
                event.exchange, event.instrument,
                book.best_bid.price, book.best_ask.price
            );
        }
        other => {
            println!(
                "[OTHER]  {:>15} {:>20} | kind={}",
                event.exchange, event.instrument, other.kind_name()
            );
        }
    }

    count += 1;
    if count >= 10 { break; }
}

println!("\nDone! Consumed {count} events from multiple exchanges.");

## 3. Selecting Individual Exchange Streams

Instead of `select_all()`, you can select a specific exchange's stream using
`streams.select(ExchangeId)`. This is useful when you need exchange-specific
processing pipelines.

In [ ]:
// Example: selecting only BinanceSpot events
// let binance_stream = streams.select(ExchangeId::BinanceSpot);
//
// This returns only events from the BinanceSpot exchange,
// allowing exchange-specific processing, rate limiting, or routing.

println!("Tip: Use streams.select(ExchangeId::BinanceSpot) to get a single exchange's stream.");
println!("Tip: Use streams.select_all() to merge all exchanges into one stream.");

## Supported Exchanges

| Exchange | Spot | Perpetual | Futures | Options |
|----------|------|-----------|---------|---------|
| Binance | `BinanceSpot` | `BinanceFuturesUsd` | — | — |
| OKX | `Okx` | `Okx` | — | `Okx` |
| Coinbase | `Coinbase` | — | — | — |
| Kraken | `Kraken` | — | — | — |
| Bybit | — | `BybitPerpetualsUsd` | — | — |
| Bitfinex | `Bitfinex` | — | — | — |
| Gateio | `GateioSpot` | `GateioPerpetualsUsd` | `GateioFuturesUsd` | `GateioOptions` |
| Bitmex | — | `Bitmex` | — | — |

### DataKind Variants

```rust
pub enum DataKind {
    Trade(PublicTrade),
    OrderBookL1(OrderBookL1),
    OrderBookL2(OrderBookL2),
    // ... more as needed
}
```

---
## Part 4: Low-Level WebSocket Integration


## 1. Core Traits Overview

| Trait | Signature | Purpose |
|-------|-----------|--------|
| `Transformer` | `Input → Vec<Result<Output, Error>>` | Stateful data transformation pipeline |
| `Validator` | `&State → Result<(), Error>` | Validate conditions on state |
| `Unrecoverable` | `&self → bool` | Classify errors as fatal or transient |
| `Terminal` | `&self → bool` | Mark events that end a stream's lifecycle |

These traits compose to build the full data pipeline:
```
WebSocket bytes → Deserialise → Validate → Transform → Output Stream
```

## 2. The Transformer Trait

A `Transformer` converts parsed exchange messages into your desired output model.
It can hold state (e.g., running totals, sequence numbers, order book snapshots).

In [ ]:
use barter_integration::{Transformer, error::SocketError};
use serde::Deserialize;

/// Raw Binance message — could be a subscription confirmation or a trade
#[derive(Debug, Deserialize)]
#[serde(untagged)]
enum BinanceMessage {
    SubResponse {
        result: Option<Vec<String>>,
        id: u32,
    },
    Trade {
        #[serde(rename = "s")]
        symbol: String,
        #[serde(rename = "p")]
        price: String,
        #[serde(rename = "q")]
        quantity: String,
        #[serde(rename = "m")]
        is_buyer_maker: bool,
    },
}

/// Our desired output: a simplified trade summary
#[derive(Debug)]
struct TradeSummary {
    symbol: String,
    price: f64,
    quantity: f64,
    side: &'static str,
    trade_count: u64,
}

/// Stateful transformer that counts trades and converts raw messages
struct TradeTransformer {
    trade_count: u64,
}

impl Transformer for TradeTransformer {
    type Error = SocketError;
    type Input = BinanceMessage;
    type Output = TradeSummary;
    type OutputIter = Vec<Result<Self::Output, Self::Error>>;

    fn transform(&mut self, input: Self::Input) -> Self::OutputIter {
        match input {
            BinanceMessage::SubResponse { .. } => {
                // Subscription confirmations produce no output
                vec![]
            }
            BinanceMessage::Trade { symbol, price, quantity, is_buyer_maker } => {
                self.trade_count += 1;
                vec![Ok(TradeSummary {
                    symbol,
                    price: price.parse().unwrap_or(0.0),
                    quantity: quantity.parse().unwrap_or(0.0),
                    side: if is_buyer_maker { "Sell" } else { "Buy" },
                    trade_count: self.trade_count,
                })]
            }
        }
    }
}

println!("TradeTransformer defined.");
println!("  Input:  BinanceMessage (raw JSON)");
println!("  Output: TradeSummary (normalised)");
println!("  State:  trade_count (running counter)");

## 3. ExchangeStream — Putting It Together

An `ExchangeStream` wraps a WebSocket connection with a `Transformer` to produce
a standard `futures::Stream` of your output type. This is the core building block
used by `barter-data` for all exchange integrations.

```rust
type ExchangeWsStream<T> = ExchangeStream<WebSocketSerdeParser, WebSocket, T>;
```

In [ ]:
use barter_integration::{
    protocol::websocket::{WebSocket, WebSocketSerdeParser, WsMessage},
    stream::ExchangeStream,
};
use futures::{SinkExt, StreamExt};
use std::collections::VecDeque;
use tokio_tungstenite::connect_async;

// Type alias for a WebSocket-based ExchangeStream
type ExchangeWsStream<T> = ExchangeStream<WebSocketSerdeParser, WebSocket, T>;

// Connect to Binance WebSocket
let mut ws_conn = connect_async("wss://fstream.binance.com/ws")
    .await
    .map(|(ws, _)| ws)
    .expect("failed to connect to Binance");

// Subscribe to BTC/USDT aggregated trades
ws_conn
    .send(WsMessage::text(
        serde_json::json!({
            "method": "SUBSCRIBE",
            "params": ["btcusdt@aggTrade"],
            "id": 1
        }).to_string()
    ))
    .await
    .expect("failed to subscribe");

// Create the ExchangeStream with our transformer
let transformer = TradeTransformer { trade_count: 0 };
let mut stream = ExchangeWsStream::new(ws_conn, transformer, VecDeque::new());

println!("Connected! Reading first 5 transformed trades...\n");

let mut count = 0;
while let Some(result) = stream.next().await {
    match result {
        Ok(summary) => {
            println!(
                "  #{} {} {} {:.2} @ {:.2}",
                summary.trade_count, summary.symbol,
                summary.side, summary.quantity, summary.price
            );
            count += 1;
            if count >= 5 { break; }
        }
        Err(e) => eprintln!("  Error: {e:?}"),
    }
}

println!("\nDone! Transformer processed {count} trades with running state.");

## 4. Validator Trait

Validators check conditions on data before it enters the transformation pipeline.
This is used internally by exchange integrations to validate subscription
responses, sequence numbers, and message integrity.

In [ ]:
use barter_integration::Validator;

/// Example: validate that a trade has positive quantity
#[derive(Debug)]
struct PositiveQuantity;

impl Validator for PositiveQuantity {
    type Context = f64; // the quantity to validate
    type Error = String;

    fn validate(&self, quantity: &Self::Context) -> Result<(), Self::Error> {
        if *quantity > 0.0 {
            Ok(())
        } else {
            Err(format!("Invalid quantity: {quantity}"))
        }
    }
}

let validator = PositiveQuantity;
assert!(validator.validate(&1.5).is_ok());
assert!(validator.validate(&-0.1).is_err());

println!("Validator example: PositiveQuantity");
println!("  validate(1.5)  → Ok");
println!("  validate(-0.1) → Err");

## 5. Error Classification: Unrecoverable & Terminal

Barter distinguishes between transient and fatal errors at the type level:

- **`Unrecoverable`**: Is this error fatal? Should we shut down?
- **`Terminal`**: Has the stream ended? Should we reconnect?

This drives the reconnection and shutdown logic in the engine.

In [ ]:
use barter_integration::{Unrecoverable, Terminal};

// Example error that can be either recoverable or not
#[derive(Debug)]
enum StreamError {
    /// Transient: can retry
    Timeout,
    /// Fatal: must stop
    AuthFailed,
    /// Stream ended naturally
    Disconnected,
}

impl Unrecoverable for StreamError {
    fn is_unrecoverable(&self) -> bool {
        matches!(self, StreamError::AuthFailed)
    }
}

impl Terminal for StreamError {
    fn is_terminal(&self) -> bool {
        matches!(self, StreamError::Disconnected | StreamError::AuthFailed)
    }
}

let errors = [StreamError::Timeout, StreamError::AuthFailed, StreamError::Disconnected];
for err in &errors {
    println!(
        "  {:?}: unrecoverable={}, terminal={}",
        err, err.is_unrecoverable(), err.is_terminal()
    );
}

## Integration Architecture

```
                          barter-integration
┌──────────────────────────────────────────────────────────┐
│                                                          │
│  WebSocket / REST                                        │
│       │                                                  │
│       ▼                                                  │
│  ┌──────────┐    ┌───────────┐    ┌──────────────┐      │
│  │  Parser   │───▶│ Validator  │───▶│ Transformer   │     │
│  │ (deser)   │    │ (check)   │    │ (stateful)    │     │
│  └──────────┘    └───────────┘    └──────┬────────┘     │
│                                          │              │
│                                          ▼              │
│                                   ExchangeStream        │
│                                  (futures::Stream)      │
└──────────────────────────────────────────────────────────┘
                         │
                         ▼
                    barter-data
              (exchange implementations)
```

### When to Use barter-integration Directly

- Adding support for a **new exchange** not yet in barter-data
- Building a **custom data pipeline** outside the Barter engine
- Implementing **private WebSocket feeds** (account updates, order status)
- Creating **REST API clients** for order management